In [1]:
import pandas as pd

In [2]:
parts = [
    "../data/split/part_1.csv",
    "../data/split/part_2.csv",
    "../data/split/part_3.csv",
    "../data/split/part_4.csv"
]
samples = []
for part in parts:
    sample = pd.read_csv(part, usecols=["owner", "name", "combined_text"])
    sample = sample.sample(frac=0.0625,random_state=42)#Changing the df size to 1M beacuse of i dont have enough memory
    #Update -> changed size to 250k due to not good results
    samples.append(sample)
df = pd.concat(samples,ignore_index=True)


In [3]:
df.shape

(261068, 3)

In [4]:
#If i run the cosine  on this data it will be very big and impossible to do beacuse of it can jump upto TB size.

In [5]:
#creating the vector data of this df

In [6]:
import joblib
vectorizer = joblib.load('../data/models/tfidf_vectorizer.pkl')
data_matrix = vectorizer.transform(df['combined_text'])  #only this column goes into TF-IDF

In [7]:
#So we got vector data on the full dataset

In [8]:
data_matrix.nnz

2990104

In [9]:
data_matrix.shape

(261068, 115158)

In [10]:
data_matrix.dtype

dtype('float64')

In [11]:
print(f"Memory usage: {data_matrix.data.nbytes / 1e6:.2f} MB")

Memory usage: 23.92 MB


In [12]:
#TF-IDF are higher dimensional
#TruncatedSVD reduces the dimension and make it dense so the FAISS can work efficently

In [13]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=700,random_state=42)
reduced_matrix = svd.fit_transform(data_matrix)

In [14]:
reduced_matrix.shape

(261068, 700)

In [15]:
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.4f}") #This tells you how much information were captured into the compressed version


Explained variance: 0.4163


In [16]:
#In here using dimension = 100 only gave as 24%. so not good

#I changed the dataset size to 1M 

#Now we are getting upto 40% is good

In [17]:
import psutil
print(f"Total RAM: {psutil.virtual_memory().total / 1e9:.2f} GB")
print(f"Available RAM: {psutil.virtual_memory().available / 1e9:.2f} GB")

Total RAM: 16.85 GB
Available RAM: 5.01 GB


In [18]:
joblib.dump(svd,"../data/models/svd_model.pkl")
joblib.dump(reduced_matrix,"../data/models/reduced_matrix.pkl")

['../data/models/reduced_matrix.pkl']

In [19]:
import joblib
reduced_matrix = joblib.load("../data/models/reduced_matrix.pkl")

In [20]:
import faiss
import numpy as np

vectors = reduced_matrix.astype(np.float32)
faiss.normalize_L2(vectors)

In [21]:
d = vectors.shape[1]
nlist = 1000

quantizer = faiss.IndexFlatIP(d)
index = faiss.IndexIVFFlat(quantizer,d,nlist,faiss.METRIC_INNER_PRODUCT)

In [22]:
index.train(vectors)
index.add(vectors)

In [23]:
index.nprobe = 10

In [24]:
import pandas as pd
indices = pd.Series(df.index, index=df['name']).drop_duplicates()

In [25]:
tfidf = joblib.load("../data/models/tfidf_vectorizer.pkl")

In [26]:
svd = joblib.load("../data/models/svd_model.pkl")

In [27]:
def search_repos(query_text, top_n=5):
    #  vectorize the query using the SAME tfidf vectorizer
    query_tfidf = tfidf.transform([query_text])
    
    # reduce using the SAME fitted SVD model
    query_reduced = svd.transform(query_tfidf).astype(np.float32)

    faiss.normalize_L2(query_reduced)

    D, I = index.search(query_reduced, top_n)
    
    return df[['owner', 'name']].iloc[I[0]]

In [28]:
print(search_repos("hospital management using python or js"))

                    owner                       name
64832            lava1201   Hotel-management-using-c
232618    ranjith-kamaraj     loan-management-system
121105            subaa23           Hotel-management
88058               tsgrp                        HPI
10515   Orange-OpenSource  opnfv-cloudify-clearwater


In [29]:
print(df['combined_text'].iloc[64832])

Hotel-management-using-c  C C


In [30]:
print(df['combined_text'].sample(10).tolist())

['android-widget A sample android app that displays a collection-view widget Java Java', 'Unsupervised_learning 无监督学习是机器学习中重要一部分。 Python Python', 'twitter-nodes A tool for extracting network relations from Twitter. Python Python', 'sub-clash A Subscribe Convert Tool for Clash Python Python', 'pgedge-northwind pgEdge Northwind Demo TypeScript TypeScript CSS JavaScript cloudflare-pages cloudflare-workers nextjs nodejs postgres react typescript pgedge', 'MarkovianTraining  Python Python TeX BibTeX Style Shell', 'postcss-high-contrast Create high contrast version of your project with ease. JavaScript JavaScript contrast postcss-plugins css contrast-plugins accessibility postcss', 'zabbix_api 用python调用zabbix api，实现自动管理zabbix监控系统 Python Python', 'godot-neural-network Machine learning in godot. To solve non-linear problems, we need to connect several perceptrons together to create a neural network. In the example, the multilayer perceptron resolve a XOR operation. GDScript GDScript godot-engi

In [31]:
print('hospital' in tfidf.vocabulary_)


True


In [32]:
matches = df[df['combined_text'].str.contains('hospital', case=False, na=False)]
print(matches.shape)
print(matches[['owner', 'name']].head(10))

(105, 3)
                      owner                               name
22241                alipay                         RJU_Ant_QA
23353  open-power-workgroup                           Hospital
25744                agueye                     Matlabu-Chifai
26428           small-bears                          hospitals
27608          MoH-Malaysia                     covid19-public
31481           girishsaraf  Online-Appointment-Booking-System
39713           HospitalRun                         components
42655            Trustroots                         trustroots
44531           small-bears                          hospitalq
45921        prateeksinghal         Hospital-Management-System


In [37]:
# Get TF-IDF + SVD vector for a KNOWN relevant repo
known_idx = df[df['name'] == 'Hospital-Management-System'].index[0]
known_vec = vectors[known_idx:known_idx+1]

# Compare directly to your query vector
from sklearn.metrics.pairwise import cosine_similarity
query_tfidf = tfidf.transform(["hospital management using python or js"])
query_reduced = svd.transform(query_tfidf).astype(np.float32)
faiss.normalize_L2(query_reduced)

similarity = cosine_similarity(query_reduced, known_vec)
print(similarity)

[[0.29110524]]


In [34]:
#only 40% information in the compression makes the prediction not good
#So reducing the size to 250k from 1M

In [35]:
1 -2

-1